# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [ ]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
import sys
import utils

load_dotenv(override=True)

MODEL = "gpt-4.1-nano"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [ ]:
# Inspired by LangChain's Document - let's have something similar

# 定義一個 Result 類別，用來表示最終處理後的結果
# 這裡參考了 LangChain 的 Document 結構
class Result(BaseModel):
    page_content: str
    metadata: dict

In [ ]:
# A class to perfectly represent a chunk

# 定義一個 chunk 類別，代表一個分段後的文件區塊
class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query") # headline：每個區塊的標題，通常是幾個字或詞，方便搜尋或顯示
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions") # summary：該區塊的摘要，用幾句話說明主要內容
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way") # original_text：該區塊的原始文字，不做修改

    # 將 Chunk 轉換成 Result 物件（加上 metadata）
    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)

# 定義一個 Chunks 類別，包含多個 Chunk
class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [ ]:
# # 了解程式碼運作過程
# documents = []

# for folder in KNOWLEDGE_BASE_PATH.iterdir():
#     print(f"📌folder = {folder}")
#     doc_type = folder.name
#     print(f"folder.name = {folder.name}")
    
#     for file in folder.rglob("*.md"):
#         with open(file, "r", encoding="utf-8") as f:
#             documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

# print(f"Loaded {len(documents)} documents")


In [ ]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [ ]:
documents = fetch_documents()

In [ ]:
documents[0].keys()

In [ ]:
# 查看 document[0] 是什麼

for key, value in documents[0].items():
    print("\033[34m" + str(key) + "\033[0m:" + str(value))

### Donezo! On to Step 2 - make the chunks

In [ ]:
def make_prompt(document):
    # 根據文件的長度來估算應該切成幾個 chunk（平均每個 chunk 約 500 字元）
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    # 回傳一個多行字串（f-string），作為給模型的提示詞（prompt）
    # 模型會根據這個 prompt 來幫文件分割成多個 chunks
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [ ]:
print(make_prompt(documents[0]))

In [ ]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [ ]:
# 查看 make_messages 的用法

for key, value in make_messages(documents[0])[0].items():
    print("\033[34m" + str(key) + "\033[0m:" + str(value))

In [ ]:
# 查看 make_messages 的用法

msg = make_messages(documents[0])[0]

print("{")
for i, (key, value) in enumerate(msg.items()):
    comma = "," if i < len(msg) - 1 else ""
    print(f"  \033[34m'{key}'\033[0m: {repr(value)}{comma}")
print("}")

<span style="color: yellow;">------------查看 llm 的輸出內容------------</span>

In [ ]:
messages = make_messages(documents[0])  
response = completion(model=MODEL, messages=messages, response_format=Chunks)

In [ ]:
utils.view_model_response(response)

In [ ]:
utils.view_model_response(response.choices[0].message.content)

In [ ]:
reply = response.choices[0].message.content

In [ ]:
doc_as_chunks = Chunks.model_validate_json(reply).chunks

In [ ]:
utils.view_model_response(doc_as_chunks)

In [ ]:
utils.view_model_response(doc_as_chunks[0])

In [ ]:
# 用結構化的查看模型輸出
utils.view_model_response(doc_as_chunks[0].as_result(documents[0]))

<span style="color: yellow;">------------查看 llm 的輸出內容------------</span>

In [ ]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [ ]:
process_document(documents[0])

In [ ]:
# 用結構化的方式察看結果
utils.view_model_response(process_document(documents[0]))

In [ ]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [ ]:
chunks = create_chunks(documents)

In [ ]:
print(len(chunks))

In [ ]:
chunks[0]

In [ ]:
chunks[0].metadata

### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [ ]:
def create_embeddings(chunks):
    # 建立（或連接）一個 Chroma 向量資料庫客戶端
    # PersistentClient(path=DB_NAME) 表示資料會儲存在本地硬碟的指定資料夾中
    chroma = PersistentClient(path=DB_NAME)
    
    # 如果資料庫中已經存在同名的 collection（資料集合），就先刪除它
    # 這樣可以確保每次重新建立資料庫時都是乾淨的
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    # 從 chunks 中取出每個 chunk 的文字內容（page_content）
    # 這些文字將用來生成向量（embeddings）
    texts = [chunk.page_content for chunk in chunks]
    
    # 呼叫 OpenAI 的 embedding 模型，把文字轉換成向量表示
    # - model=embedding_model：指定使用的 embedding 模型（例如 text-embedding-3-large）
    # - input=texts：一次把所有文字送進模型產生 embeddings
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    
    # 從回傳結果中取出實際的 embedding 向量（即每段文字的數值表示）    
    vectors = [e.embedding for e in emb]

    # 在 Chroma 資料庫中建立或取得一個 collection（相當於一個資料表）
    # collection_name 是你指定的集合名稱（例如 "docs"）
    collection = chroma.get_or_create_collection(collection_name)

    # 為每個 chunk 生成唯一的 ID（以索引位置轉成字串）
    ids = [str(i) for i in range(len(chunks))]
    
    # 取出每個 chunk 的 metadata（例如來源檔案路徑與類型）
    metas = [chunk.metadata for chunk in chunks]

    # 將所有資料新增進 Chroma 資料庫：
    # - ids：每筆資料的唯一識別碼
    # - embeddings：文字對應的向量
    # - documents：原始文字內容
    # - metadatas：附加資訊（來源與文件類型）
    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [ ]:
create_embeddings(chunks)

# Nothing more to do here... right?

Wait! Didja think I'd forget??

從向量資料庫中直接讀取資料

In [ ]:
chroma = PersistentClient(path=DB_NAME) # 建立一個持久化的 Chroma 客戶端，用於連接或創建一個本地向量資料庫，存放位置為 DB_NAME 指定的路徑
collection = chroma.get_or_create_collection(collection_name) # 從資料庫中獲取指定名稱的 collection（集合），如果不存在就創建一個新的
result = collection.get(include=['embeddings', 'documents', 'metadatas']) # 從集合中取出所有記錄，並指定要包含的欄位：embeddings（向量）、documents（文件內容）、metadatas（元數據）
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [ ]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [ ]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [ ]:
system_prompt = """
您是一名文件重新排序器。 您將獲得一個問題以及從知識庫查詢中檢索到的相關文本片段列表。 
這些片段按照檢索順序提供；這應該大致按照相關性排序，但您可能可以進一步改進排序。 
您必須根據問題的相關性對提供的片段進行排序，將最相關的片段排在最前面。
僅回覆重新排序後的片段 ID 列表，其他內容一律不包括。請包含您獲得的所有片段 ID，並重新排序。
"""

user_prompt = f"""
使用者提出了以下問題： 
{question}  
請根據與該問題的相關程度，將所有文字區塊（chunks）從最相關到最不相關進行排序。  
請包含提供給你的所有區塊 ID，並依重新排序後的順序列出。

這些是區塊
"""

for index, chunk in enumerate(chunks):
    user_prompt += f"# CHUNK ID: {index + 1}: \n\n{question}\n\nOr"

In [ ]:
def rerank(question, chunks):
    # 定義一個名為 rerank 的函式，用來重新排序（rerank）文件片段（chunks）
    # 參數：
    # - question：使用者提出的問題
    # - chunks：一組與問題相關的文本片段，每個 chunk 代表一段內容
    
    
    # 定義 system_prompt，作為「系統提示詞」(system message)
    # 這段文字告訴模型它的角色是「文件重新排序器」，要根據問題重新排列 chunks 的相關性
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""

    # 定義 user_prompt，作為「使用者提示詞」(user message)
    # 內容包含使用者的問題，並要求模型依照相關性重新排列所有 chunks
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    
    # 在提示詞中加入說明：接下來是所有的 chunks
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    
    
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    # 將模型回覆解析成 RankOrder 物件，並取出 order 屬性（這應該是一個整數列表，例如 [2, 1, 3]）
    # 表示 chunk 的新排序順序
    order = RankOrder.model_validate_json(reply).order
    print(order)
    
    # 根據模型提供的新順序（order），重排 chunks
    # 注意：因為 CHUNK ID 從 1 開始編號，而 Python 的索引從 0 開始，所以用 i - 1
    # 最後返回重新排序後的 chunks
    return [chunks[i - 1] for i in order]

In [ ]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    """
    基於向量相似度的無排序上下文檢索函數
    
    功能說明：
    此函數接受一個自然語言問題，將其轉換為向量表示，然後在向量資料庫中檢索最相關的文檔片段。
    函數使用語義相似度匹配，而非關鍵字匹配，因此能理解問題的語義含義。
    
    參數：
    question : str
        要查詢的自然語言問題，例如："Who won the IIOTY award?"
    
    返回值：
    list
        包含檢索結果的列表，每個元素為Result物件，具有以下屬性：
        - page_content: str - 檢索到的文本片段
        - metadata: dict - 相關元資料，包含來源文件路徑和類型
        
    工作流程：
    1. 將輸入問題透過OpenAI嵌入模型轉換為3072維向量
    2. 在ChromaDB向量資料庫中查詢與問題向量最相似的K個文檔片段
    3. 將查詢結果封裝為Result物件列表並返回
    
    使用範例：
    >>> question = "Who won the IIOTY award?"
    >>> chunks = fetch_context_unranked(question)
    >>> for chunk in chunks:
    >>>     print(f"內容: {chunk.page_content[:100]}...")
    >>>     print(f"來源: {chunk.metadata['source']}")
    
    注意事項：
    - 函數依賴外部變數：embedding_model, collection
    - RETRIEVAL_K控制檢索數量，預設為10個片段
    - 此為無排序檢索，結果按向量相似度從高到低排列
    - 確保OpenAI API金鑰和ChromaDB連接已正確設定
    
    異常處理：
    - 若嵌入模型呼叫失敗，將引發OpenAI API異常
    - 若向量資料庫查詢失敗，將引發ChromaDB異常
    """
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding  # 3072 維的向量
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [ ]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [ ]:
utils.view_model_response(chunks)


In [ ]:
utils.view_model_response(chunks[0])

In [ ]:
for chunk in chunks:
    print(chunk.page_content[:15]+"...")

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for chunk in reranked:
    print(chunk.page_content[:15]+"...")

In [ ]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked = rerank(question, chunks)

In [ ]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

In [ ]:
reranked[0].page_content

In [ ]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [ ]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [ ]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [ ]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [ ]:
rewrite_query("Who won the IIOTY award?", [])

In [ ]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [ ]:
answer_question("Who won the IIOTY award?", [])

In [ ]:
answer_question("Who went to Manchester University?", [])